In [ ]:
# IMPORT LIBRARIES
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import Counter

from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    ConfusionMatrixDisplay
)

from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_extraction.text import CountVectorizer
from textblob import TextBlob
from nltk.util import ngrams
from wordcloud import WordCloud
import warnings
warnings.filterwarnings("ignore")
# 1. DATASET COLLECTION
print(" ")
print("1. DATASET COLLECTION")
print(" ")
df = pd.read_csv("sentiment\creditcard.csv")
print("\nDataset Loaded Successfully!\n")
print("Shape :", df.shape)
print("\nFirst Five Rows\n")
print(df.head())
print("\nDataset Information\n")
df.info()

print("\nMissing Values\n")
print(df.isnull().sum())
print("\nClass Distribution\n")
print(df["Class"].value_counts())

# 2. NLP PREPROCESSING
print(" ")
print("2. NLP PREPROCESSING")
print(" ")
# Convert rows into text
def convert_to_sentence(row):
    return f"Transaction amount {row['Amount']} at time {row['Time']} class {row['Class']}"
df["Text"] = df.apply(convert_to_sentence, axis=1)

# Cleaning
def clean_text(text):
    text = text.lower()
    text = re.sub(r"[^a-z0-9 ]", "", text)
    text = re.sub(r"\s+", " ", text)
    return text.strip()
df["Clean_Text"] = df["Text"].apply(clean_text)
print("\nSample Cleaned Text\n")
print(df["Clean_Text"].head())

# 3. AUTOCOMPLETE
print(" ")
print("3. AUTOCOMPLETE")
print(" ")

# Sample data for faster processing
sample_df = df.sample(10000, random_state=42)
tokens = " ".join(sample_df["Clean_Text"]).split()
bigram_list = list(ngrams(tokens, 2))
autocomplete_model = {}
for w1, w2 in bigram_list:
    autocomplete_model.setdefault(w1, []).append(w2)

for word in autocomplete_model:
    autocomplete_model[word] = Counter(autocomplete_model[word])

def autocomplete(word):

    if word in autocomplete_model:
        return autocomplete_model[word].most_common(5)

    return []

print("\nAutocomplete Suggestions\n")
for word in ["transaction", "amount", "time", "class"]:
    print(word, ":", autocomplete(word))

# 4. AUTOCORRECT
print(" ")
print("4. AUTOCORRECT")
print(" ")

misspelled_words = ["transction","amunt","clas","tim"]
for word in misspelled_words:
    print(word, "->", TextBlob(word).correct())

# 5. PERFORMANCE METRICS
print(" ")
print("5. PERFORMANCE METRICS")
print(" ")
X = df.drop(columns=["Class", "Text", "Clean_Text"])
y = df["Class"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42
)

# Logistic Regression

lr = LogisticRegression(max_iter=1000)
lr.fit(X_train, y_train)
pred_lr = lr.predict(X_test)
print("\nLogistic Regression\n")
print("Accuracy :", accuracy_score(y_test, pred_lr))
print("Precision:", precision_score(y_test, pred_lr))
print("Recall   :", recall_score(y_test, pred_lr))
print("F1 Score :", f1_score(y_test, pred_lr))

# 6. USER EXPERIENCE
print(" ")
print("6. USER EXPERIENCE")
print(" ")
feedback = pd.DataFrame({

    "User":
    ["U1","U2","U3","U4","U5","U6","U7","U8","U9","U10"],

    "Rating":
    [5,4,5,3,4,5,4,5,5,4]

})

print(feedback)
plt.figure(figsize=(6,4))
feedback["Rating"].value_counts().sort_index().plot(kind="bar")
plt.title("User Feedback")
plt.xlabel("Rating")
plt.ylabel("Number of Users")
plt.show()

# 7. ALGORITHM COMPARISON
print(" ")
print("7. ALGORITHM COMPARISON")
print(" ")
rf = RandomForestClassifier(
    n_estimators=100,
    random_state=42
)

rf.fit(X_train, y_train)
pred_rf = rf.predict(X_test)
comparison = pd.DataFrame({

    "Algorithm":
    ["Logistic Regression","Random Forest"],

    "Accuracy":
    [
        accuracy_score(y_test,pred_lr),
        accuracy_score(y_test,pred_rf)
    ],

    "Precision":
    [
        precision_score(y_test,pred_lr),
        precision_score(y_test,pred_rf)
    ],

    "Recall":
    [
        recall_score(y_test,pred_lr),
        recall_score(y_test,pred_rf)
    ],

    "F1 Score":
    [
        f1_score(y_test,pred_lr),
        f1_score(y_test,pred_rf)
    ]

})
print(comparison)

# 8. VISUALIZATION
print(" ")
print("8. VISUALIZATION")
print(" ")
# Fraud Distribution
plt.figure(figsize=(6,4))
df["Class"].value_counts().plot(kind="bar")
plt.title("Fraud vs Non-Fraud")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

# Amount Distribution
plt.figure(figsize=(8,5))
plt.hist(df["Amount"], bins=50)
plt.title("Transaction Amount Distribution")
plt.xlabel("Amount")
plt.ylabel("Frequency")
plt.show()

# Correlation Matrix
plt.figure(figsize=(12,10))
sns.heatmap(df.drop(columns=["Text","Clean_Text"]).corr(),
            cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show()

# Word Cloud

text = " ".join(sample_df["Clean_Text"])

wordcloud = WordCloud(
    width=900,
    height=500,
    background_color="white"
).generate(text)
plt.figure(figsize=(12,6))
plt.imshow(wordcloud)
plt.axis("off")
plt.title("Word Cloud")
plt.show()

# Confusion Matrix

cm = confusion_matrix(y_test, pred_rf)
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot()
plt.title("Random Forest Confusion Matrix")
plt.show()

# Top Words

print("\nTop 10 Frequent Words\n")
vectorizer = CountVectorizer(stop_words="english")
X_words = vectorizer.fit_transform(sample_df["Clean_Text"])
word_freq = np.asarray(X_words.sum(axis=0)).ravel()
words = vectorizer.get_feature_names_out()
freq = pd.DataFrame({

    "Word": words,
    "Frequency": word_freq

})

freq = freq.sort_values(
    by="Frequency",
    ascending=False
)

print(freq.head(10))

plt.figure(figsize=(8,5))
plt.bar(freq.head(10)["Word"],
        freq.head(10)["Frequency"])
plt.xticks(rotation=45)
plt.title("Top 10 Frequent Words")
plt.show()
